# Treinar a voz da Tamaki (Piper) · grátis

Este notebook pega o pacote do **Gravador de voz** (`tamaki_dataset.zip`) e treina uma voz Piper própria, que depois roda **offline** no Termux e no editor de voz.

Ele parte de uma voz brasileira já treinada (`pt_BR-faber-medium`) e só ajusta para o seu timbre (*fine-tuning*). Assim a dicção em português já começa boa e o treino fica bem mais curto.

**No Colab** (mais fácil): *Ambiente de execução > Alterar o tipo de ambiente > GPU T4* e depois *Executar tudo*. Ele vai pedir o .zip e salvar o progresso no seu Google Drive.

**No Kaggle** (30 h de GPU grátis por semana): *File > Import Notebook* com este arquivo. No painel da direita, ligue *Accelerator: GPU T4* e *Internet: on* (precisa do celular verificado na conta). Envie o .zip em *Add Input > Upload* (vira um Dataset). Para treinar sem deixar a aba aberta, use *Save Version > Save & Run All*.

Tempo: com 150 frases e 4000 passos, cerca de 1 a 2 horas numa T4. Se cair no meio, rode tudo de novo: ele continua de onde parou.

In [ ]:
# ===== Ajustes =====
NOME_VOZ = "tamaki"          # o arquivo final vira pt_BR-tamaki-medium.onnx
PASSOS = 4000                # mais passos = mais parecido com o seu áudio (3000 a 6000 é o normal)
LOTE = 16                    # baixe para 8 se der "CUDA out of memory"
BASE = "pt/pt_BR/faber/medium"  # voz de partida (checkpoint do rhasspy/piper-checkpoints)
SALVAR_NO_DRIVE = True       # só no Colab: guarda o progresso no Google Drive
PIPER_VERSAO = "v1.8.0"      # a mesma que o Termux usa para pronunciar

import os, glob, subprocess, sys
NO_KAGGLE = os.path.isdir("/kaggle")
NO_COLAB = "google.colab" in sys.modules
TRAB = "/kaggle/working" if NO_KAGGLE else "/content"
SAIDA = f"{TRAB}/voz_{NOME_VOZ}"
if NO_COLAB and SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAIDA = f"/content/drive/MyDrive/voz_{NOME_VOZ}"
os.makedirs(SAIDA, exist_ok=True)
print("Onde:", "Kaggle" if NO_KAGGLE else "Colab" if NO_COLAB else "outro", "| progresso em:", SAIDA)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "SEM GPU: ligue a GPU antes de continuar"

## 1. Instalar o Piper (uns 5 minutos)

In [ ]:
!apt-get -qq update && apt-get -qq install -y build-essential cmake ninja-build > /dev/null
%cd {TRAB}
if not os.path.isdir("piper1-gpl"):
    !git clone -q --depth 1 --branch {PIPER_VERSAO} https://github.com/OHF-Voice/piper1-gpl.git
%cd {TRAB}/piper1-gpl
!pip install -q -e '.[train]'
!./build_monotonic_align.sh
!python setup.py -q build_ext --inplace
!python -c "import piper.train.vits.monotonic_align as m; print('Piper pronto')"

## 2. O pacote de frases

In [ ]:
import zipfile, csv, soundfile as sf
DADOS = f"{TRAB}/dataset"
if not os.path.exists(f"{DADOS}/metadata.csv"):
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob(f"{TRAB}/*.zip")
    pastas = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/metadata.csv", recursive=True)]
    if pastas:  # o Kaggle às vezes já descompacta o .zip enviado
        DADOS = pastas[0]
    else:
        if not zips and NO_COLAB:
            from google.colab import files
            print("Escolha o tamaki_dataset.zip (o do Gravador de voz):")
            zips = [os.path.abspath(n) for n in files.upload()]
        if not zips:
            raise SystemExit("Não achei o tamaki_dataset.zip. No Kaggle: Add Input > Upload.")
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(DADOS)

PASTA_WAV = f"{DADOS}/wavs" if os.path.isdir(f"{DADOS}/wavs") else DADOS
linhas, faltando, seg = [], [], 0.0
with open(f"{DADOS}/metadata.csv", encoding="utf-8") as f:
    for l in csv.reader(f, delimiter="|"):
        if len(l) < 2 or not l[1].strip():
            continue
        cam = os.path.join(PASTA_WAV, l[0])
        if not os.path.exists(cam):
            faltando.append(l[0]); continue
        seg += sf.info(cam).duration
        linhas.append(l)
METADADOS = f"{TRAB}/metadata_ok.csv"
with open(METADADOS, "w", encoding="utf-8") as f:
    f.writelines(f"{a}|{t.strip()}\n" for a, t, *_ in linhas)
print(f"{len(linhas)} frases, {seg/60:.1f} min de áudio." + (f" Faltando {len(faltando)} arquivos." if faltando else ""))
if len(linhas) < 50:
    print("Aviso: com menos de 50 frases a voz fica instável. O ideal é 150 ou mais.")

## 3. Voz de partida

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
CKPT_BASE = f"{TRAB}/base.ckpt"
if not os.path.exists(CKPT_BASE):
    arquivos = HfApi().list_repo_files("rhasspy/piper-checkpoints", repo_type="dataset")
    ckpts = [a for a in arquivos if a.startswith(BASE + "/") and a.endswith(".ckpt")]
    if not ckpts:
        raise SystemExit(f"Não achei checkpoint em {BASE}. Opções pt_BR: " +
                         ", ".join(sorted({os.path.dirname(a) for a in arquivos if a.startswith('pt/pt_BR') and a.endswith('.ckpt')})))
    print("Baixando", ckpts[-1])
    os.replace(hf_hub_download("rhasspy/piper-checkpoints", ckpts[-1], repo_type="dataset", local_dir=f"{TRAB}/hf"), CKPT_BASE)
print("Base:", CKPT_BASE)

## 4. Treinar

A linha `[warmstart] Copied ... parameters` tem que mostrar um número grande (centenas). Se cair, rode de novo: ele continua do `last.ckpt`.

In [ ]:
import math
por_epoca = max(1, math.ceil(len(linhas) * 0.9 / LOTE))
EPOCAS = max(50, math.ceil(PASSOS / por_epoca))
VAL = max(1, round(250 / por_epoca))  # testa e salva o progresso a cada ~250 passos
ultimos = sorted(glob.glob(f"{SAIDA}/lightning_logs/*/checkpoints/last.ckpt"), key=os.path.getmtime)
continuar = f"--ckpt_path {ultimos[-1]}" if ultimos else f"--model.warmstart_ckpt {CKPT_BASE}"
print(f"{EPOCAS} épocas ({por_epoca} passos cada).", "Continuando de " + ultimos[-1] if ultimos else "Começando do faber.")
!python -m piper.train fit \
  --data.voice_name "{NOME_VOZ}" \
  --data.csv_path "{METADADOS}" \
  --data.audio_dir "{PASTA_WAV}" \
  --model.sample_rate 22050 \
  --data.espeak_voice "pt-br" \
  --data.cache_dir "{TRAB}/cache" \
  --data.config_path "{SAIDA}/config.json" \
  --data.batch_size {LOTE} \
  --data.num_workers 2 \
  --trainer.accelerator gpu --trainer.devices 1 \
  --trainer.max_epochs {EPOCAS} \
  --trainer.check_val_every_n_epoch {VAL} \
  --trainer.default_root_dir "{SAIDA}" \
  {continuar}

## 5. Exportar e ouvir

In [ ]:
import shutil
from IPython.display import Audio, display
ckpts = sorted(glob.glob(f"{SAIDA}/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)
ESCOLHIDO = [c for c in ckpts if c.endswith("last.ckpt")][-1]   # troque por outro da lista se soar melhor
print("\n".join(ckpts)); print("Exportando:", ESCOLHIDO)
FINAL = f"{SAIDA}/pt_BR-{NOME_VOZ}-medium.onnx"
!python -m piper.train.export_onnx --checkpoint "{ESCOLHIDO}" --output-file "{FINAL}"
shutil.copyfile(f"{SAIDA}/config.json", FINAL + ".json")

import wave
from piper import PiperVoice
voz = PiperVoice.load(FINAL)
for i, t in enumerate(["Gente, você não tem noção do que aconteceu!",
                       "Nem relógio trabalha de graça, amiga. Nos vemos no próximo sábado, beijão!"]):
    with wave.open(f"{TRAB}/teste{i}.wav", "wb") as w:
        voz.synthesize_wav(t, w)
    print(t); display(Audio(f"{TRAB}/teste{i}.wav"))

## 6. Baixar a voz

Ponha o `voz_tamaki_piper.zip` no celular em `Download/tamaki_personagem/voz-treinada` e rode no Termux:

```
voz --importar
```

No editor de voz: *Voz > Termux · Kokoro ou voz treinada > Tamaki treinada (Piper)*.

In [ ]:
PACOTE = f"{TRAB}/voz_{NOME_VOZ}_piper.zip"
with zipfile.ZipFile(PACOTE, "w") as z:
    z.write(FINAL, os.path.basename(FINAL))
    z.write(FINAL + ".json", os.path.basename(FINAL) + ".json")
print("Pronto:", PACOTE, f"({os.path.getsize(PACOTE)/1e6:.0f} MB)")
if NO_COLAB:
    from google.colab import files
    files.download(PACOTE)
elif NO_KAGGLE:
    print("No Kaggle: baixe pelo painel Output (à direita) ou pela aba Output da versão salva.")